# Entscheidsuche.ch API - Example Notebook

This notebook demonstrates the `entscheidsuche` Python package for accessing Swiss court decisions from entscheidsuche.ch endpoints.

This package is independently developed and is not official, endorsed by, associated with, or affiliated with entscheidsuche.ch.


## 1. Setup and Installation


In [1]:
# Run `uv sync` from the repo root and select the repo `.venv` as this notebook's kernel.
import entscheidsuche

print(f"entscheidsuche {entscheidsuche.__version__} ready")

entscheidsuche 0.1.0 ready


In [2]:
# Import the package
from collections import defaultdict

from entscheidsuche import EntscheidsucheClient
from entscheidsuche import utils

# Create the client
client = EntscheidsucheClient()
print("Client created successfully!")

Client created successfully!


## 2. Basic Information

### 2.1 List Available Cantons


In [3]:
# List all cantons
cantons = client.list_cantons()
print("Available cantons:")
for code, name in sorted(cantons.items()):
    print(f"  {code}: {name}")

Available cantons:
  AG: Aargau
  AI: Appenzell Innerrhoden
  AR: Appenzell Ausserrhoden
  BE: Bern
  BL: Basel-Land
  BS: Basel-Stadt
  CH: Bund
  FR: Freiburg
  GE: Genf
  GL: Glarus
  GR: Graubünden
  JU: Jura
  LU: Luzern
  NE: Neuenburg
  NW: Nidwalden
  OW: Obwalden
  SG: St. Gallen
  SH: Schaffhausen
  SO: Solothurn
  SZ: Schwyz
  TG: Thurgau
  TI: Tessin
  UR: Uri
  VD: Waadt
  VS: Wallis
  ZG: Zug
  ZH: Zürich


### 2.2 List Available Scrapers


In [4]:
# List all known scrapers
scrapers = client.list_scrapers()
print(f"Number of scrapers: {len(scrapers)}")
print("\nScrapers by canton:")

# Group by canton
by_canton = defaultdict(list)
for s in scrapers:
    canton = s.split("_")[0]
    by_canton[canton].append(s)

for canton in sorted(by_canton.keys()):
    print(f"  {canton}: {', '.join(by_canton[canton])}")

Number of scrapers: 40

Scrapers by canton:
  AG: AG_Weitere
  AI: AI_Aktuell, AI_Bericht
  AR: AR_Gerichte
  BE: BE_ZivilStraf, BE_Anwaltsaufsicht, BE_Verwaltungsgericht, BE_Steuerrekurs, BE_BVD, BE_Weitere
  BL: BL_Gerichte
  BS: BS_Omni
  CH: CH_BGE, CH_BVGer, CH_BPatG
  FR: FR_Gerichte
  GE: GE_Gerichte
  GL: GL_Omni
  GR: GR_Gerichte
  JU: JU_Gerichte
  LU: LU_Gerichte
  NW: NW_Gerichte
  OW: OW_Gerichte
  SG: SG_Gerichte, SG_Publikationen
  SO: SO_Omni
  SZ: SZ_Gerichte
  TG: TG_OG
  TI: TI_Gerichte
  UR: UR_Gerichte
  VD: VD_FindInfo, VD_Omni
  VS: VS_Gerichte
  XX: XX_Upload
  ZG: ZG_Verwaltungsgericht
  ZH: ZH_Obergericht, ZH_Verwaltungsgericht, ZH_Steuerrekurs, ZH_Baurekurs, ZH_Sozialversicherungsgericht


## 3. Search API

### 3.1 Simple Text Search


In [5]:
# Simple search
results = client.search("Mietvertrag", size=5)

print(f"Found {results.total} results (showing first {len(results.hits)})")
print(f"Search took {results.took_ms}ms\n")

for hit in results.hits:
    print(f"📄 {hit.signatur}")
    print(f"   Date: {hit.date} | Language: {hit.language} | Score: {hit.score:.2f}")
    title = hit.title.get("de", "N/A")
    print(f"   Title: {title[:70]}..." if len(title) > 70 else f"   Title: {title}")
    print()

Found 8110 results (showing first 5)
Search took 40ms

📄 VD_TC_010_XG09-009299_nodate
   Date: 2021-01-01 | Language: fr | Score: 15.27
   Title: Vaud Tribunal cantonal Chambre des recours civile XG09.009299

📄 GE_CJ_004_C-2371-2014_2014-09-08
   Date: 2014-09-08 | Language: fr | Score: 14.69
   Title: Genève Cour de Justice (Cour civile) Chambre des baux et loyers 08.09....

📄 GE_CJ_004_C-3071-2015_2016-11-07
   Date: 2016-11-07 | Language: fr | Score: 13.35
   Title: Genève Cour de Justice (Cour civile) Chambre des baux et loyers 07.11....

📄 GE_CJ_004_C-15351-2011_2014-08-06
   Date: 2014-08-06 | Language: fr | Score: 13.11
   Title: Genève Cour de Justice (Cour civile) Chambre des baux et loyers 06.08....

📄 TI_TRAC_004_16-2021-50_2022-02-02
   Date: 2022-02-02 | Language: it | Score: 12.81
   Title: Ticino Tribunale di appello diritto civile La Camera civile dei reclam...



### 3.2 Search with Filters


In [6]:
# Search with filters
results = client.search(
    query="Arbeitsvertrag Kündigung",
    size=10,
    canton="ZH",  # Only Zürich
    language="de",  # Only German
    date_from="2020-01-01",  # From 2020
    date_to="2024-12-31",  # To 2024
)

print(f"Found {results.total} results from Zürich in German (2020-2024)\n")

for hit in results.hits:
    print(f"📄 {hit.signatur} ({hit.date})")

Found 364 results from Zürich in German (2020-2024)

📄 ZH_OG_001_LA230017_2023-10-19 (2023-10-19)
📄 ZH_OG_001_LA230008_2024-03-25 (2024-03-25)
📄 ZH_OG_001_LA210031_2022-12-16 (2022-12-16)
📄 ZH_VG_001_-VB-2022-00120_2022-09-21 (2022-09-21)
📄 ZH_OG_999_AH230077-L_2024-01-25 (2024-01-25)
📄 ZH_VG_001_-VB-2022-00059_2022-11-10 (2022-11-10)
📄 ZH_OG_001_LA190012_2020-01-14 (2020-01-14)
📄 ZH_VG_001_-VB-2020-00380_2020-10-27 (2020-10-27)
📄 ZH_OG_001_LF220039_2022-08-05 (2022-08-05)
📄 ZH_OG_999_AN210020_2022-11-22 (2022-11-22)


### 3.3 Search with Sorting


In [7]:
# Search sorted by date (newest first)
results = client.search(
    query="Erbrecht",
    size=5,
    sort_by="date",
    sort_order="desc",
)

print("Newest Erbrecht decisions:")
for hit in results.hits:
    print(f"  {hit.date}: {hit.signatur}")

Newest Erbrecht decisions:
  2026-04-22: GR_KG_006_ZR1-2026-9_2026-04-22
  2026-04-13: SZ_KG_002_ZK2-2026-22_2026-04-13
  2026-03-17: AG_OG_001_ZOR-2025-36_2026-03-17
  2026-03-02: GR_KG_006_ZR1-2026-3_2026-03-02
  2026-02-06: SZ_VG_002_II-2025-55_2026-02-06


### 3.4 Pagination


In [8]:
# Pagination example
page_size = 5

# Page 1
page1 = client.search("Strafrecht", size=page_size, from_=0)
print(f"Page 1 (results 1-{len(page1.hits)} of {page1.total}):")
for hit in page1.hits:
    print(f"  - {hit.signatur}")

# Page 2
page2 = client.search("Strafrecht", size=page_size, from_=page_size)
print(f"\nPage 2 (results {page_size + 1}-{page_size + len(page2.hits)}):")
for hit in page2.hits:
    print(f"  - {hit.signatur}")

Page 1 (results 1-5 of 10000):
  - CH_BVGE_001_BVGE-2007-32_2007-08-08
  - ZH_SRK_001_DB-2009-27_2009-11-02
  - CH_BGE_002_BGE-103-IA-95_1977-03-30
  - VD_TC_031_CCST-2016-0008_2017-05-10
  - BL_KG_004_460-16-134_2016-11-22

Page 2 (results 6-10):
  - BL_KG_004_460-15-138_2016-03-15
  - BL_KG_004_460-16-119_2016-09-06
  - BL_KG_004_460-15-137_2015-09-22
  - BL_KG_004_460-2014-40_2014-11-17
  - CH_BGE_001_BGE-88-I-93_1962-07-04


### 3.5 Raw Elasticsearch Query


In [9]:
# Advanced: raw Elasticsearch query
raw_query = {
    "size": 5,
    "query": {
        "bool": {
            "must": [{"match": {"abstract.de": "Datenschutz"}}],
            "filter": [{"exists": {"field": "attachment.language"}}],
        }
    },
    "_source": ["title", "date", "hierarchy", "abstract"],
    "highlight": {"fields": {"abstract.de": {}}},
}

raw_results = client.search_raw(raw_query)
print(f"Raw query returned {raw_results['hits']['total']} results")
if raw_results["hits"]["hits"]:
    first = raw_results["hits"]["hits"][0]
    print(f"First result ID: {first['_id']}")
    print(f"First result source: {first['_source']}")

Raw query returned {'value': 512, 'relation': 'eq'} results
First result ID: SG_VSG_001_IV-2009-312_2010-06-14
First result source: {'date': '2010-06-14', 'hierarchy': ['SG', 'SG_VSG', 'SG_VSG_001'], 'abstract': {'de': 'Art. 33 ATSG, Art. 50a AHVG. Schweigepflicht, Datenschutz. Als Beistand ist der Beschwerdeführer vorliegend gesetzlich ermächtigt, den verbeiständeten Versicherten zu vertreten. Daher muss er sich nicht durch eine vom Versicherten unterzeichnete Vollmacht als Vertreter legitimieren. Die Verwaltung kann sich ihm gegenüber nicht auf die Schweigepflicht bzw. den Datenschutz berufen (Entscheid des Versicherungsgerichts des Kantons St. Galle vom 14. Juni 2010, IV 2009/312).', 'it': 'Art. 33 ATSG, Art. 50a AHVG. Schweigepflicht, Datenschutz. Als Beistand ist der Beschwerdeführer vorliegend gesetzlich ermächtigt, den verbeiständeten Versicherten zu vertreten. Daher muss er sich nicht durch eine vom Versicherten unterzeichnete Vollmacht als Vertreter legitimieren. Die Verwaltun

## 4. Document Access

### 4.1 Get Document Metadata (JSON)


In [10]:
# First, get a document signatur from search
results = client.search("BGE", size=1, spider="CH_BGE")

if results.hits:
    hit = results.hits[0]
    print(f"Getting document: {hit.signatur} from {hit.spider}")

    # Get full JSON metadata
    doc = client.get_document_json(hit.spider, hit.signatur)

    print("\nDocument details:")
    print(f"  Signatur: {doc.signatur}")
    print(f"  Spider: {doc.spider}")
    print(f"  Date: {doc.date}")
    print(f"  Language: {doc.language}")
    print(f"  Case numbers: {doc.num}")
    print(f"  Original URL: {doc.url}")
    print(f"  HTML URL: {doc.html_url}")
    print(f"  PDF URL: {doc.pdf_url}")
    print(f"  Kopfzeile (DE): {doc.kopfzeile.get('de', 'N/A')[:100]}...")
else:
    print("No documents found")

Getting document: CH_BGE_001_BGE-86-I-226_1960-10-20 from CH_BGE

Document details:
  Signatur: CH_BGE_001
  Spider: CH_BGE
  Date: 1960-10-20
  Language: fr
  Case numbers: ['BGE 86 I 226']
  Original URL: https://search.bger.ch/ext/eurospider/live/de/php/clir/http/index.php?highlight_docid=atf%3A%2F%2F86-I-226%3Ade&lang=de&zoom=&type=show_document
  HTML URL: https://entscheidsuche.ch/docs/CH_BGE/CH_BGE_001_BGE-86-I-226_1960-10-20.html
  PDF URL: None
  Kopfzeile (DE): Bundesgericht (BGE) Teil I 20.10.1960 BGE 86 I 226...


### 4.2 Get Document HTML


In [11]:
# Get HTML content (if available)
if results.hits:
    hit = results.hits[0]
    try:
        html = client.get_document_html(hit.spider, hit.signatur)
        print(f"HTML content length: {len(html)} characters")
        print("\nFirst 500 characters:")
        print(html[:500])
    except Exception as e:
        print(f"HTML not available: {e}")

HTML content length: 5947 characters

First 500 characters:
<!DOCTYPE html><html lang="fr"><head><meta charset="utf-8"/></head><body><div class="content">
      
  <a name="idp275536"></a><div class="big bold">Urteilskopf</div>
<br>86 I 226<br><br><br><div class="paraatf"> 31. Extrait de l'arrêt du 20 octobre 1960 dans la cause Rey contre Valais, Commission de revision en matière d'expropriation.</div>
<div class="paraatf"></div>
  <a name="idp276928"></a>
  <a name="idp279056"></a><br><div id="regeste" lang="de">
<div class="big bold">Regeste</div>
<br>


### 4.3 Download Documents


In [12]:
from pathlib import Path

# Download documents to local directory
if results.hits:
    hit = results.hits[0]
    output_dir = Path("./downloads")

    saved = client.download_document(
        spider=hit.spider,
        signatur=hit.signatur,
        output_dir=output_dir,
        formats=["json"],  # Only JSON to avoid large downloads in example
    )

    print("Downloaded files:")
    for fmt, path in saved.items():
        print(f"  {fmt}: {path}")

Downloaded files:
  json: downloads/CH_BGE_001_BGE-86-I-226_1960-10-20.json


## 5. Index and Jobs Files

### 5.1 Get Index File


In [13]:
# Get latest index for a scraper
index = client.get_index("CH_BGE")

print("Index for CH_BGE:")
print(f"  Job type: {index.jobtyp}")
print(f"  Time: {index.time}")
print(f"  Total documents: {index.total_count}")
print(f"  New documents: {len(index.new_documents)}")
print(f"  Updated documents: {len(index.updated_documents)}")
print(f"  Deleted documents: {len(index.deleted_documents)}")

Index for CH_BGE:
  Job type: update
  Time: 2026-05-23_02:14:27
  Total documents: 21259
  New documents: 0
  Updated documents: 0
  Deleted documents: 0


### 5.2 Get Jobs File


In [14]:
# Get latest jobs file for a scraper
jobs = client.get_jobs("CH_BGE")

print("Jobs for CH_BGE:")
print(f"  Job type: {jobs.jobtyp}")
print(f"  Time: {jobs.time}")
print(f"  Number of signatures: {len(jobs.documents)}")

Jobs for CH_BGE:
  Job type: update
  Time: 2026-05-23_02:14:27
  Number of signatures: 8


## 6. Blocklist


In [15]:
# Get blocklist of removed documents
try:
    blocklist = client.get_blocklist()
    print(f"Blocked documents: {len(blocklist)}")
    if blocklist:
        print(f"First 5: {blocklist[:5]}")
except Exception as e:
    print(f"Could not fetch blocklist: {e}")

Blocked documents: 66
First 5: ['VD_TC_013_PE23-022276_nodate', 'VD_TC_003_Ord---2012---7_2012-03-22', 'VD_TC_004_Arr-t---2024---890_nodate', 'VD_TC_017_Arr-t---2011---1149_2011-10-05', 'VD_TC_002_HC---2025---24_nodate']


## 7. Sitemap


In [16]:
# Get sitemap index
sitemaps = client.get_sitemap_index()

print(f"Number of sitemaps: {len(sitemaps)}")
print("\nFirst 10 sitemaps:")
for sm in sitemaps[:10]:
    print(f"  {sm}")

Number of sitemaps: 63

First 10 sitemaps:
  https://entscheidsuche.ch/docs/Sitemaps/ZH_Obergericht_1.xml
  https://entscheidsuche.ch/docs/Sitemaps/GL_Omni_1.xml
  https://entscheidsuche.ch/docs/Sitemaps/SZ_Gerichte_1.xml
  https://entscheidsuche.ch/docs/Sitemaps/NE_Omni_1.xml
  https://entscheidsuche.ch/docs/Sitemaps/GE_Gerichte_4.xml
  https://entscheidsuche.ch/docs/Sitemaps/SO_Omni_1.xml
  https://entscheidsuche.ch/docs/Sitemaps/ZH_Steuerrekurs_1.xml
  https://entscheidsuche.ch/docs/Sitemaps/GE_Gerichte_3.xml
  https://entscheidsuche.ch/docs/Sitemaps/CH_BGer_1.xml
  https://entscheidsuche.ch/docs/Sitemaps/CH_BVGer_2.xml


## 8. Utility Functions

### 8.1 Parse Case Numbers


In [17]:
# Parse Swiss case numbers
case_numbers = [
    "4A_123/2023",  # Federal Court
    "A-1234/2022",  # Federal Administrative Court
    "VB.2023.00123",  # Cantonal court
]

for cn in case_numbers:
    parsed = utils.parse_case_number(cn)
    print(f"\n{cn}:")
    for key, value in parsed.items():
        print(f"  {key}: {value}")


4A_123/2023:
  court: BGer
  chamber: 4A
  number: 123
  year: 2023
  original: 4A_123/2023

A-1234/2022:
  court: BVGer
  chamber: A
  number: 1234
  year: 2022
  original: A-1234/2022

VB.2023.00123:
  court: None
  chamber: None
  number: None
  year: 2023
  original: VB.2023.00123


### 8.2 Build Search Queries


In [18]:
import json

# Build a search query programmatically
query = utils.build_search_query(
    text="Vertragsrecht",
    canton="BE",
    year=2023,
    language="de",
)

print("Generated Elasticsearch query:")
print(json.dumps(query, indent=2))

Generated Elasticsearch query:
{
  "query": {
    "bool": {
      "must": [
        {
          "query_string": {
            "query": "Vertragsrecht",
            "default_operator": "AND"
          }
        }
      ],
      "filter": [
        {
          "term": {
            "hierarchy": "BE"
          }
        },
        {
          "range": {
            "date": {
              "gte": "2023-01-01",
              "lte": "2023-12-31"
            }
          }
        },
        {
          "term": {
            "attachment.language": "de"
          }
        }
      ]
    }
  }
}


### 8.3 Convert Search Results to DataFrame


In [19]:
# Convert to pandas DataFrame (requires pandas)
try:
    results = client.search("Haftung", size=10)
    df = utils.search_results_to_dataframe(results)

    print(f"DataFrame with {len(df)} rows:")
    display(df[["signatur", "date", "language", "score"]].head())
except ImportError:
    print("pandas not installed. Install with: uv add pandas")

DataFrame with 10 rows:


,signatur,date,language,score
0,CH_BVGE_001_BVGE-2010-4_2009-12-14,2009-12-14,et,14.676387
1,CH_BVGE_001_A-926-2009_2009-03-02,2009-03-02,de,12.768400
2,ZH_OG_001_LB210024_2021-11-09,2021-11-09,de,12.440386
3,ZH_OG_001_LB220017_2022-09-30,2022-09-30,de,12.440386
4,CH_BVGE_001_A-6121-2008_2010-09-06,2010-09-06,de,12.341928


### 8.4 Extract Text from HTML


In [20]:
# Extract plain text from HTML
sample_html = """
<html>
<body>
<h1>Urteil vom 15. März 2023</h1>
<p>Der Beschwerdeführer macht geltend, dass...</p>
<p>Die Vorinstanz hat zu Recht festgestellt, dass...</p>
</body>
</html>
"""

text = utils.extract_text_from_html(sample_html)
print("Extracted text:")
print(text)

Extracted text:
Urteil vom 15. März 2023 Der Beschwerdeführer macht geltend, dass... Die Vorinstanz hat zu Recht festgestellt, dass...


### 8.5 Group and Filter Documents


In [21]:
# Get some documents
results = client.search("Recht", size=50)
documents = [hit.to_case_document() for hit in results.hits]

# Group by canton
by_canton = utils.group_by_canton(documents)
print("Documents by canton:")
for canton, docs in sorted(by_canton.items(), key=lambda x: -len(x[1]))[:5]:
    print(f"  {canton}: {len(docs)} documents")

# Group by year
by_year = utils.group_by_year(documents)
print("\nDocuments by year:")
for year, docs in sorted(by_year.items(), reverse=True)[:5]:
    print(f"  {year}: {len(docs)} documents")

Documents by canton:
  CH: 22 documents
  VD: 19 documents
  GE: 8 documents
  VS: 1 documents

Documents by year:
  2023: 1 documents
  2022: 2 documents
  2021: 17 documents
  2020: 1 documents
  2019: 3 documents


### 8.6 Get Statistics


In [22]:
# Get statistics for a set of documents
stats = utils.get_statistics(documents)

print(f"Statistics for {stats['total']} documents:")
print(f"  Cantons covered: {stats['cantons_count']}")
print(f"  Years span: {stats['years_span']}")
print(f"\n  By language: {stats['by_language']}")

Statistics for 50 documents:
  Cantons covered: 4
  Years span: ('1955', '2023')

  By language: {'de': 14, 'fr': 36}


## 9. Context Manager Usage


In [23]:
# Use as context manager for proper cleanup
with EntscheidsucheClient() as client:
    results = client.search("Bundesgericht", size=3)
    print(f"Found {results.total} results")

print("Client properly closed")

Found 10000 results
Client properly closed


## 10. Rate Limiting

The client includes built-in rate limiting to be respectful to the server. You can customize it:


In [24]:
import time

# Create client with custom rate limiting
client = EntscheidsucheClient(
    timeout=30.0,
    rate_limit_delay=1.0,  # 1 second between requests
)

# Multiple requests will be spaced out
start = time.time()
for i in range(3):
    results = client.search("Test", size=1)
    print(f"Request {i + 1} at {time.time() - start:.1f}s")

client.close()

Request 1 at 0.2s
Request 2 at 1.2s
Request 3 at 2.3s


## Summary

This package provides:

1. **Search API** - Full Elasticsearch support with filters and pagination
2. **Document Access** - JSON, HTML, and PDF download
3. **Index/Jobs Files** - Track scraper updates
4. **Blocklist** - Check removed documents
5. **Sitemaps** - Browse all content
6. **Utilities** - Parse case numbers, convert to DataFrame, filter/group documents

Remember to be respectful of the server - use rate limiting and avoid excessive requests.
